# Prompting guide: the Prompt Optimization Pipeline

**Contributor: [Meenakshi Mangipudi](https://www.linkedin.com/in/meenakshimangipudi/), Masters student, Christ University.**
Developed with Prof. Abhishek Kumar Jha, SPJIMR. More teaching resources: https://abhishekkumar7.github.io/personal-website/#resources/prompting

Describe a task in plain words. The notebook:
1. says what kind of task it is,
2. picks the 3 best of 14 prompting techniques, and
3. rewrites your task as a ready-to-paste prompt for each, then saves a PDF report.

**You need:** a free Gemini API key from Google AI Studio (https://aistudio.google.com/apikey).
**How to run:** click *Runtime > Run all*, paste your key when asked, then change the task in Step 4 and run again.
Do not paste confidential or personal data.

## Step 1. Install the libraries

In [ ]:
!pip install -q google-genai reportlab

## Step 2. Add your Gemini API key
If you saved the key in Colab *Secrets* as `GEMINI_API_KEY`, it is used automatically. Otherwise paste it when asked; it is not shown or saved.

In [ ]:
import getpass
GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    pass
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass('Enter your Gemini API key: ')

## Step 3. Set up the model and the 14-technique framework
The notebook tries `gemini-3.8-flash` first and falls back to `gemini-3.5-flash-lite`. If the model is busy it waits and retries.

In [ ]:
import json, re, time
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)
MODELS = ["gemini-3.8-flash", "gemini-3.5-flash-lite"]   # first choice, then fallback

def ask_gemini(text):
    """Send one prompt; retry when the model is busy, try the fallback model if the first is unavailable."""
    last_error = None
    for name in MODELS:
        for attempt in range(4):
            try:
                return client.models.generate_content(model=name, contents=text).text
            except Exception as e:
                last_error, msg = e, str(e)
                if "404" in msg or "NOT_FOUND" in msg:
                    break                                   # model not available for this key: try the next one
                if any(c in msg for c in ("429", "500", "503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "overloaded")):
                    time.sleep(2 * (2 ** attempt))          # busy: wait 2, 4, 8, 16 seconds
                    continue
                raise                                       # wrong key or other error: stop and show it
    raise last_error

TECHNIQUES = """
1. Zero-Shot Prompting - Straightforward, standard tasks where the expected output is clear: summarising, translating, factual Q&A, classification, recommendations.
2. Few-Shot Prompting - The task has a style, format or logic that is hard to describe in words. Show 2 to 5 examples first.
3. Contrastive Prompting - Outputs keep drifting the wrong way. Show a good example and a bad one to make the target clear.
4. Chain-of-Thought - Complex problems that need logic, maths or several reasoning steps.
5. Step-by-Step Prompting - Procedures, instructions, workflows and how-to guides where order matters.
6. Tree of Thought - Open-ended problems with several valid approaches. Explore branches, evaluate them, then converge.
7. Self-Consistency - Accuracy is critical. Run several reasoning paths and compare where they agree.
8. Structural Prompts - The format of the output matters: tables, JSON, a set document layout. Define the shape before the content.
9. Persona Prompting - The perspective or expertise behind the answer matters. Assign a role or identity.
10. Reverse Prompting - You want to learn from an example or copy a style: reconstruct the prompt that produced an output.
11. AI Interview Technique - Complex deliverables that need context. Let the model ask questions before it produces anything.
12. Thought Provocation - Challenge assumptions, stress-test an idea, find blind spots. Ask the model to argue against or critique.
13. Meta-Prompting - You want to improve or generate prompts themselves. The model writes or refines the prompt.
14. Iterative Refinement - High-stakes writing. Draft, critique, then revise in sequence.
"""

def analyze_prompt(user_prompt):
    analysis_prompt = f"""
You are a prompt engineering expert. A person with no knowledge of prompting has given you a task in plain language.
Your job is to analyze it and select the top 3 most suitable prompting techniques from the list below.

TECHNIQUES AND WHEN TO USE THEM:
{TECHNIQUES}

USER'S RAW INPUT:
{user_prompt}

First write 2-3 sentences explaining what kind of task this is and why certain techniques suit it.
Then return a JSON block (and nothing after it) in this exact format:

EXPLANATION: <your 2-3 sentence explanation here>

```json
{{
  "top3": [
    {{"rank": 1, "technique": "technique name", "reason": "one sentence explaining why this fits best"}},
    {{"rank": 2, "technique": "technique name", "reason": "one sentence explaining why this fits"}},
    {{"rank": 3, "technique": "technique name", "reason": "one sentence explaining why this fits"}}
  ]
}}
```
"""
    raw = ask_gemini(analysis_prompt).strip()
    explanation = ""
    if "EXPLANATION:" in raw:
        explanation = raw.split("EXPLANATION:")[1].split("```")[0].strip()
    match = re.search(r"```json\s*(\{.*?\})\s*```", raw, re.S) or re.search(r"(\{.*\})", raw, re.S)
    data = json.loads(match.group(1))
    return explanation, data["top3"]

def rewrite_prompt(user_prompt, technique):
    rewrite_text = f"""
You are a prompt engineering expert. A person with no knowledge of prompting gave you this task in plain language:

"{user_prompt}"

Rewrite this as a well-structured prompt using the "{technique}" technique.
Apply all the rules of this technique properly.
The rewritten prompt should be ready to paste directly into any AI tool.
Output only the rewritten prompt. No explanation, no preamble, no labels.
"""
    return ask_gemini(rewrite_text).strip()

def run_pipeline(user_prompt):
    print("Analyzing your input...\n")
    explanation, top3 = analyze_prompt(user_prompt)
    print("Task Analysis:")
    print(explanation)
    print("\nTop 3 Techniques Selected:")
    for item in top3:
        star = "  BEST CHOICE" if item["rank"] == 1 else ""
        print(f"  #{item['rank']} {item['technique']}{star}")
        print(f"      {item['reason']}")
    print("\nGenerating optimized prompts...\n")
    results = []
    for item in top3:
        rewritten = rewrite_prompt(user_prompt, item["technique"])
        results.append({**item, "rewritten_prompt": rewritten})
        label = " (Best Choice)" if item["rank"] == 1 else ""
        print("=" * 60)
        print(f"Type {item['rank']} - {item['technique']}{label}")
        print(f"Why: {item['reason']}")
        print(f"\nOptimized Prompt:\n{rewritten}\n")
    return explanation, results

## Step 4. Describe your task in plain words
Change the text between the triple quotes and run this cell. Examples: *"Write interview questions for a junior data analyst"*, *"Explain why our margin fell this quarter"*.

In [ ]:
raw_prompt = """
Searching the best web series on Amazon Prime for college group.
"""

explanation, results = run_pipeline(raw_prompt)

## Step 5. Save the result as a PDF report

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable, Table, TableStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from xml.sax.saxutils import escape

def save_to_pdf(explanation, results, raw_prompt, filename="prompt_pipeline_output.pdf"):
    doc = SimpleDocTemplate(filename, pagesize=A4, rightMargin=2*cm, leftMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm)
    title = ParagraphStyle("T", fontSize=18, fontName="Helvetica-Bold", spaceAfter=6, textColor=colors.HexColor("#1a1a2e"))
    sub = ParagraphStyle("S", fontSize=10, fontName="Helvetica", spaceAfter=16, textColor=colors.HexColor("#666666"))
    label = ParagraphStyle("L", fontSize=9, fontName="Helvetica-Bold", textColor=colors.HexColor("#888888"), spaceAfter=4, spaceBefore=16)
    inp = ParagraphStyle("I", fontSize=12, fontName="Helvetica-Bold", textColor=colors.HexColor("#1a1a2e"), spaceAfter=8, backColor=colors.HexColor("#f5f5f5"), borderPadding=8)
    body = ParagraphStyle("B", fontSize=10, fontName="Helvetica", textColor=colors.HexColor("#444444"), spaceAfter=6, leading=15)
    head_best = ParagraphStyle("HB", fontSize=13, fontName="Helvetica-Bold", textColor=colors.white)
    head = ParagraphStyle("H", fontSize=13, fontName="Helvetica-Bold", textColor=colors.HexColor("#1a1a2e"))
    reason = ParagraphStyle("R", fontSize=9, fontName="Helvetica-Oblique", textColor=colors.HexColor("#666666"), spaceAfter=8, spaceBefore=4)
    prompt = ParagraphStyle("P", fontSize=10, fontName="Helvetica", textColor=colors.HexColor("#1a1a2e"), leading=15)

    story = [Paragraph("Prompt Optimization Pipeline", title),
             Paragraph("Powered by Gemini · Based on the 14-Technique Prompting Framework · Contributor: Meenakshi Mangipudi", sub),
             HRFlowable(width="100%", thickness=1, color=colors.HexColor("#e0e0e0"), spaceAfter=12),
             Paragraph("YOUR INPUT", label), Paragraph(escape(raw_prompt.strip()), inp), Spacer(1, 0.2*cm),
             Paragraph("TASK ANALYSIS", label), Paragraph(escape(explanation), body), Spacer(1, 0.3*cm),
             HRFlowable(width="100%", thickness=1, color=colors.HexColor("#e0e0e0"), spaceAfter=12),
             Paragraph("TOP 3 RECOMMENDED PROMPT TYPES", label), Spacer(1, 0.2*cm)]
    for r in results:
        best = r["rank"] == 1
        text = f"Type {r['rank']}  -  {escape(r['technique'])}" + ("   ✓  Best Choice" if best else "")
        bar = Table([[Paragraph(text, head_best if best else head)]], colWidths=[16.5*cm])
        bar.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#1a1a2e") if best else colors.HexColor("#f0f0f0")),
                                 ("TOPPADDING", (0, 0), (-1, -1), 10), ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
                                 ("LEFTPADDING", (0, 0), (-1, -1), 12), ("RIGHTPADDING", (0, 0), (-1, -1), 12)]))
        story += [bar, Paragraph(f"Why this technique: {escape(r['reason'])}", reason)]
        box = Table([[Paragraph(escape(r["rewritten_prompt"]).replace("\n", "<br/>"), prompt)]], colWidths=[16.5*cm])
        box.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#fafafa")), ("BOX", (0, 0), (-1, -1), 0.5, colors.HexColor("#dddddd")),
                                 ("TOPPADDING", (0, 0), (-1, -1), 10), ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
                                 ("LEFTPADDING", (0, 0), (-1, -1), 12), ("RIGHTPADDING", (0, 0), (-1, -1), 12)]))
        story += [box, Spacer(1, 0.5*cm)]
    story += [HRFlowable(width="100%", thickness=1, color=colors.HexColor("#e0e0e0"), spaceBefore=8),
              Paragraph("Generated by the Prompt Optimization Pipeline · 14-Technique Framework · Contributor: Meenakshi Mangipudi, Christ University",
                        ParagraphStyle("F", fontSize=8, fontName="Helvetica", textColor=colors.HexColor("#aaaaaa"), spaceBefore=6))]
    doc.build(story)
    print(f"PDF saved: {filename}")

save_to_pdf(explanation, results, raw_prompt)

## Step 6. Download the PDF (Colab only)

In [ ]:
try:
    from google.colab import files
    files.download("prompt_pipeline_output.pdf")
except ImportError:
    print("Not in Colab: the PDF is saved next to this notebook.")